In [1]:
import pandas as pd
import time
import requests

url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/isoc_eb_ai?lang=EN"

response = requests.get(url)

data = response.json()

In [2]:
base_url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/isoc_eb_ai"

In [2]:
def get_eurostat_indicator(indicator_code, unit_code, column_name):

    params = {
        "lang": "EN",
        "geoLevel": "country",
        "size_emp": "GE10",
        "indic_is": indicator_code,
        "unit": unit_code,
        "sinceTimePeriod": "2021"
    }

    response = requests.get(base_url, params=params)

    print(column_name, "-", response.status_code)

    if response.status_code != 200:
        return pd.DataFrame()

    data = response.json()

    geo_index = data["dimension"]["geo"]["category"]["index"]
    geo_labels = data["dimension"]["geo"]["category"]["label"]

    time_index = data["dimension"]["time"]["category"]["index"]

    values = data["value"]

    number_of_years = len(time_index)

    records = []

    for country_code, country_position in geo_index.items():

        for year, year_position in time_index.items():

            value_position = (
                country_position * number_of_years
                + year_position
            )

            value = values.get(str(value_position))

            if value is not None:

                record = {
                    "country_code": country_code,
                    "country": geo_labels[country_code],
                    "year": int(year),
                    column_name: value
                }

                records.append(record)

    time.sleep(0.3)

    return pd.DataFrame(records)

In [4]:
adoption_df = get_eurostat_indicator(
    indicator_code="E_AI_TANY",
    unit_code="PC_ENT",
    column_name="ai_adoption_pct"
)

ai_adoption_pct - 200


In [5]:
adoption_df.head()

,country_code,country,year,ai_adoption_pct
0,BE,Belgium,2021,10.32
1,BE,Belgium,2023,13.81
2,BE,Belgium,2024,24.71
3,BE,Belgium,2025,34.54
4,BG,Bulgaria,2021,3.29


In [6]:
marketing_df = get_eurostat_indicator(
    indicator_code="E_AI_PMS",
    unit_code="PC_ENT_AI_TANY",
    column_name="ai_marketing_pct"
)

ai_marketing_pct - 200


In [7]:
adoption_df.head()

,country_code,country,year,ai_adoption_pct
0,BE,Belgium,2021,10.32
1,BE,Belgium,2023,13.81
2,BE,Belgium,2024,24.71
3,BE,Belgium,2025,34.54
4,BG,Bulgaria,2021,3.29


In [8]:
cost_df = get_eurostat_indicator(
    indicator_code="E_AI_BCST",
    unit_code="PC_ENT_AI_EC",
    column_name="high_cost_barrier_pct"
)

high_cost_barrier_pct - 200


In [9]:
skills_df = get_eurostat_indicator(
    indicator_code="E_AI_BLE",
    unit_code="PC_ENT_AI_EC",
    column_name="lack_expertise_pct"
)

lack_expertise_pct - 200


In [10]:
data_quality_df = get_eurostat_indicator(
    indicator_code="E_AI_BDDT",
    unit_code="PC_ENT_AI_EC",
    column_name="data_quality_barrier_pct"
)

data_quality_barrier_pct - 200


In [11]:
privacy_df = get_eurostat_indicator(
    indicator_code="E_AI_BCDP",
    unit_code="PC_ENT_AI_EC",
    column_name="privacy_concern_pct"
)

privacy_concern_pct - 200


In [12]:
legal_df = get_eurostat_indicator(
    indicator_code="E_AI_BLEG",
    unit_code="PC_ENT_AI_EC",
    column_name="legal_uncertainty_pct"
)

legal_uncertainty_pct - 200


In [13]:
ai_df = adoption_df.copy()

In [14]:
ai_df = ai_df.merge(
    marketing_df,
    on=["country_code", "country", "year"],
    how="outer"
)

ai_df = ai_df.merge(
    cost_df,
    on=["country_code", "country", "year"],
    how="outer"
)

ai_df = ai_df.merge(
    skills_df,
    on=["country_code", "country", "year"],
    how="outer"
)

ai_df = ai_df.merge(
    data_quality_df,
    on=["country_code", "country", "year"],
    how="outer"
)

ai_df = ai_df.merge(
    privacy_df,
    on=["country_code", "country", "year"],
    how="outer"
)

ai_df = ai_df.merge(
    legal_df,
    on=["country_code", "country", "year"],
    how="outer"
)

In [15]:
ai_df = ai_df.sort_values(
    by=["year", "ai_adoption_pct"],
    ascending=[True, False]
).reset_index(drop=True)

In [16]:
ai_df.shape

(132, 10)

In [17]:
ai_df.columns

Index(['country_code', 'country', 'year', 'ai_adoption_pct',
       'ai_marketing_pct', 'high_cost_barrier_pct', 'lack_expertise_pct',
       'data_quality_barrier_pct', 'privacy_concern_pct',
       'legal_uncertainty_pct'],
      dtype='str')

In [18]:
ai_df.isna().sum()

country_code                 0
country                      0
year                         0
ai_adoption_pct              1
ai_marketing_pct             6
high_cost_barrier_pct       44
lack_expertise_pct          43
data_quality_barrier_pct    44
privacy_concern_pct         44
legal_uncertainty_pct       44
dtype: int64

In [19]:
ai_2025 = ai_df.loc[
    ai_df["year"] == 2025
].copy()

In [22]:
ai_2025[
    ["country", "ai_adoption_pct"]
].sort_values(
    by="ai_adoption_pct",
    ascending=False
).head(10)

,country,ai_adoption_pct
99,Denmark,42.03
100,Finland,37.82
101,Sweden,35.04
102,Belgium,34.54
103,Luxembourg,33.61
104,Netherlands,33.21
105,Austria,29.95
106,Norway,28.89
107,Germany,25.97
108,Estonia,23.40


In [23]:
barrier_columns = [
    "high_cost_barrier_pct",
    "lack_expertise_pct",
    "data_quality_barrier_pct",
    "privacy_concern_pct",
    "legal_uncertainty_pct"
]

In [24]:
ai_2025[barrier_columns].mean().sort_values(
    ascending=False
)

lack_expertise_pct          69.593478
legal_uncertainty_pct       49.561739
privacy_concern_pct         48.936522
data_quality_barrier_pct    42.658261
high_cost_barrier_pct       41.346087
dtype: float64

In [25]:
adoption_by_year = (
    ai_df.groupby("year")["ai_adoption_pct"]
    .mean()
    .round(2)
)

In [26]:
adoption_by_year

year
2021     7.01
2023     7.81
2024    13.36
2025    19.27
Name: ai_adoption_pct, dtype: float64